In [1]:
# ! pip install scikit-multilearn

In [2]:
import os
import random
import functools
import csv
import numpy as np
import torch
import torch.nn.functional as F
from sklearn.metrics import f1_score, multilabel_confusion_matrix, roc_curve, auc
from skmultilearn.model_selection import iterative_train_test_split
from datasets import Dataset, DatasetDict
from peft import (
    LoraConfig,
    prepare_model_for_kbit_training,
    get_peft_model
)
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer
)
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import cycle

/root/miniconda3/envs/py310/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import torch
print("GPU 是否可", torch.cuda.is_available())
print("当前 GPU：", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "无")




GPU 是否可 True
当前 GPU： NVIDIA H800 PCIe


In [4]:
# ! pip install -U bitsandbytes

In [5]:
# ! pip install -U transformers

In [6]:
# import os
# os.environ["WANDB_DISABLED"] = "true"

# os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
# # cache_dir = "D:/hf_models"

# import os
# # os.environ['MODELSCOPE_CACHE'] = 'D:\\modelscope_cache'
# # output_dir = "D:/model"
# os.makedirs(output_dir, exist_ok=True)

In [7]:
# from google.colab import drive
# drive.mount('/content/drive')

In [8]:
# model name
# model_name = 'mistralai/Mistral-7B-v0.1'

# toke_name='deepseek-ai/DeepSeek-R1-Distill-Qwen-7B'
# model_name='deepseek-ai/DeepSeek-R1-Distill-Qwen-7B'
# model_name = 'D:/ModelScope/models/deepseek-ai/DeepSeek-R1-Distill-Llama-8B'


# model_name = "LLM-Research/Meta-Llama-3-8B-Instruct"
# toke_name= "LLM-Research/Meta-Llama-3-8B-Instruct"

# model_name = "Qwen/Qwen3-8B"
# toke_name = "Qwen/Qwen3-8B"


# model_name = "RLHFlow/ArmoRM-Llama3-8B-v0.1"
# toke_name = "RLHFlow/ArmoRM-Llama3-8B-v0.1"

model_name  = r"/root/autodl-tmp/llm_models/LLM-Research/Meta-Llama-3-70B"
# model_name ="LLM-Research/Meta-Llama-3.1-8B"
toke_name ="LLM-Research/Meta-Llama-3-70B"

In [9]:
testcsv = 'Test-Dataset(3).csv'
traincsv = 'Training-Dataset(3).csv'

In [10]:
def tokenize_examples(examples, tokenizer):
    tokenized_inputs = tokenizer(examples['text'])
    tokenized_inputs['labels'] = examples['labels']
    return tokenized_inputs

In [11]:
# define custom batch preprocessor
def collate_fn(batch, tokenizer):
    dict_keys = ['input_ids', 'attention_mask', 'labels']
    d = {k: [dic[k] for dic in batch] for k in dict_keys}
    d['input_ids'] = torch.nn.utils.rnn.pad_sequence(
        d['input_ids'], batch_first=True, padding_value=tokenizer.pad_token_id
    )
    d['attention_mask'] = torch.nn.utils.rnn.pad_sequence(
        d['attention_mask'], batch_first=True, padding_value=0
    )
    d['labels'] = torch.stack(d['labels'])
    return d

In [12]:
def plot_multilabel_roc(labels, predictions, num_classes):
    fpr = dict()
    tpr = dict()
    roc_auc = dict()
    for i in range(num_classes):
        fpr[i], tpr[i], _ = roc_curve(labels[:, i], predictions[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])

    colors = cycle(['blue', 'red', 'green', 'yellow', 'cyan', 'magenta', 'black'])
    plt.figure(figsize=(10, 8))
    for i, color in zip(range(num_classes), colors):
        plt.plot(fpr[i], tpr[i], color=color, lw=2,
                 label=f'ROC curve of class {i} (area = {roc_auc[i]:.2f})')
    plt.plot([0, 1], [0, 1], 'k--', lw=2)
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('Receiver Operating Characteristic for Multi-label')
    plt.legend(loc="lower right")
    plt.show()

In [13]:
def plot_confusion_matrix(conf_matrix, class_idx, ax, class_names=["Absent", "Present"]):
    sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', cbar=False, ax=ax)
    ax.set_xlabel('Predicted labels')
    ax.set_ylabel('True labels')
    ax.set_title(f'Class {class_idx}')
    ax.xaxis.set_ticklabels(class_names)
    ax.yaxis.set_ticklabels(class_names)

In [14]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred

    # 确保 logits 是 numpy array
    if isinstance(logits, torch.Tensor):
        logits_np = torch.sigmoid(logits).cpu().numpy()
    else:
        logits_np = 1 / (1 + np.exp(-logits))  # 如果是 raw numpy logits，用 sigmoid 转换

    predictions = logits_np > 0.5

    # Calculate F1 scores
    f1_micro = f1_score(labels, predictions, average='micro')
    f1_macro = f1_score(labels, predictions, average='macro')
    f1_weighted = f1_score(labels, predictions, average='weighted')

    # 定义统一的 eval_f1，用于 TrainingArguments 的 metric_for_best_model
    eval_f1 = f1_macro

    # Plot Confusion Matrix for each label
    conf_matrices = multilabel_confusion_matrix(labels, predictions)
    fig, ax = plt.subplots(1, len(conf_matrices), figsize=(15, 5))
    if len(conf_matrices) > 1:
        for idx, cm in enumerate(conf_matrices):
            plot_confusion_matrix(cm, idx, ax[idx])
    else:
        plot_confusion_matrix(conf_matrices[0], 0, ax)
    plt.tight_layout()
    plt.show()

    # Plot ROC Curves
    plot_multilabel_roc(labels, logits_np, num_classes=labels.shape[1])

    return {
        'eval_f1': eval_f1,           # 用于 Trainer 的 metric_for_best_model
        'eval_f1_micro': f1_micro,
        'eval_f1_macro': f1_macro,
        'eval_f1_weighted': f1_weighted
    }


In [15]:
class CustomTrainer(Trainer):
    def __init__(self, label_weights, **kwargs):
        super().__init__(**kwargs)
        self.label_weights = label_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        """
        **kwargs 接收 num_items_in_batch 或未来可能新增的参数
        """
        labels = inputs.pop("labels")

        # forward pass
        outputs = model(**inputs)
        logits = outputs.get("logits")

        # compute custom loss with label weights
        loss = F.binary_cross_entropy_with_logits(
            logits, labels.to(torch.float32), pos_weight=self.label_weights
        )
        return (loss, outputs) if return_outputs else loss


In [16]:
random.seed(0)

In [17]:
# load data
with open(traincsv, newline='', encoding="gbk") as csvfile:
    data = list(csv.reader(csvfile, delimiter=','))
    header_row = data.pop(0)
label_names = header_row[3:]


In [18]:
# reshape
random.shuffle(data)
idx, text, labels = list(zip(*[
    (
        int(row[0]),  # 索引

        # === 在输入前加入任务提示 ===
        (
            "Task: Determine whether there is a conflict between Requirement 1 and Requirement 2.\n"
            "A conflict exists if the two requirements impose incompatible, contradictory, "
            "or mutually exclusive constraints.\n\n"
            f"Requirement 1: {row[1].strip()}\n"
            f"Requirement 2: {row[2].strip()}"
        ),

        row[3:]  # 标签（多标签 / 冲突类型）
    )
    for row in data
]))
labels = np.array(labels, dtype=int)

In [19]:
labels[0]

array([0, 0, 1, 0, 0, 0, 0])

In [20]:
label_weights = 1 - labels.sum(axis=0) / labels.sum()

# stratified train test split for multilabel ds
row_ids = np.arange(len(labels))
train_idx, y_train, test_idx, y_test = iterative_train_test_split(row_ids[:,np.newaxis], labels, test_size = 0.1)

# 提取对应的文本
x_train = [str(text[i]) for i in train_idx.flatten()]
x_test = [str(text[i]) for i in test_idx.flatten()]


In [21]:
with open(testcsv, newline='', encoding="gbk") as csvfile:
    data_val = list(csv.reader(csvfile, delimiter=','))
    header_row = data_val.pop(0)
# label_names = header_row[3:]
random.shuffle(data_val)
idx_val, text_val, labels_val = list(zip(*[
    (
        int(row[0]),  # 索引
#         f"{row[1].strip()} {row[2].strip()}",  # 合并 row[1] 和 row[2] 作为文本
        f"Requirement 1: {row[1].strip()}\nRequirement 2: {row[2].strip()}",
        row[3:]  # 从第5列开始是标签
    )
    for row in data_val
]))
labels_val = np.array(labels_val, dtype=int)

label_val_weights = 1 - labels_val.sum(axis=0) / labels_val.sum()

# stratified train test split for multilabel ds
row_ids_val = np.arange(len(labels_val))
# val_idx, y_val, test_idx, y_test = iterative_train_test_split(row_ids[:,np.newaxis], labels, test_size = 0.1)
y_val= labels_val
# 提取对应的文本
# x_val = [str(text[i]) for i in idx_val.flatten()]
x_val = text_val

In [22]:
ds = DatasetDict({
    'train': Dataset.from_dict({'text': x_train, 'labels': y_train}),
    'val': Dataset.from_dict({'text': x_val, 'labels': y_val}),
    'test': Dataset.from_dict({'text': x_test, 'labels': y_test})
})

In [23]:
# preprocess dataset with tokenizer
def tokenize_examples(examples, tokenizer):
    tokenized_inputs = tokenizer(examples['text'])
    tokenized_inputs['labels'] = examples['labels']
    return tokenized_inputs

In [24]:
from modelscope import Model, AutoTokenizer, AutoModelForSequenceClassification
# from transformers import AutoModelForSequenceClassification

/root/miniconda3/envs/py310/lib/python3.10/site-packages/modelscope/utils/plugins.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [25]:
# from modelscope import AutoTokenizer
# import functools

# tokenizer = AutoTokenizer.from_pretrained(model_name)
# tokenizer.pad_token = tokenizer.eos_token

# tokenized_ds = ds.map(
#     functools.partial(tokenize_examples, tokenizer=tokenizer),
#     batched=True
# )
# tokenized_ds = tokenized_ds.with_format('torch')

tokenizer = AutoTokenizer.from_pretrained(toke_name, 
                                          # cache_dir=cache_dir,
                                          trust_remote_code=True)


tokenizer.pad_token = tokenizer.eos_token
# tokenized_ds = ds.map(functools.partial(tokenize_examples, tokenizer=tokenizer), batched=True)
tokenized_ds = ds.map(
    functools.partial(tokenize_examples, tokenizer=tokenizer),
    batched=True
)
tokenized_ds = tokenized_ds.with_format('torch')

2026-01-15 21:31:50,075 - modelscope - INFO - Target directory already exists, skipping creation.
Map: 100%|██████████| 371/371 [00:00<00:00, 17180.66 examples/s]


In [26]:
print(tokenized_ds)

DatasetDict({
    train: Dataset({
        features: ['text', 'labels', 'input_ids', 'attention_mask'],
        num_rows: 3339
    })
    val: Dataset({
        features: ['text', 'labels', 'input_ids', 'attention_mask'],
        num_rows: 379
    })
    test: Dataset({
        features: ['text', 'labels', 'input_ids', 'attention_mask'],
        num_rows: 371
    })
})


In [27]:
# tokenizer


In [28]:
# # quantization config
# quantization_config = BitsAndBytesConfig(
#     load_in_4bit=True,  # enable 4-bit quantization
#     bnb_4bit_quant_type='nf4',  # information theoretically optimal dtype for normally distributed weights
#     bnb_4bit_use_double_quant=True,  # quantize quantized weights //insert xzibit meme
#     bnb_4bit_compute_dtype=torch.bfloat16  # optimized fp format for ML
# )

In [29]:
from transformers import BitsAndBytesConfig

quantization_config = BitsAndBytesConfig(
    # load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16  # 改为 float16
)


In [30]:
labels.shape[1]

7

In [31]:
from peft import LoraConfig, get_peft_model,PrefixTuningConfig,IA3Config,PromptTuningConfig,AdaLoraConfig
# from transformers import AutoModelForSequenceClassification, AutoTokenizer, BitsAndBytesConfig
# from peft import LoraConfig, PrefixTuningConfig, IA3Config, AdapterConfig, AdapterFusionConfig, get_peft_model

# # lora config
# config = LoraConfig(
#     r=8,  # the dimension of the low-rank matrices
#     lora_alpha=16,  # scaling factor for LoRA activations vs pre-trained weight activations
#     target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
#     lora_dropout=0.2,  # dropout probability of the LoRA layers
#     # bias='none',  # whether to train bias weights, so 'none' for attention layers
#     task_type='SEQ_CLS'
# )



# config = IA3Config(
#     task_type="SEQ_CLS",
#     target_modules=["q_proj", "v_proj", "k_proj", "o_proj", "gate_proj", "down_proj", "up_proj"],
#     feedforward_modules=["gate_proj", "down_proj", "up_proj"]
# )
# config = IA3Config(
#     task_type="SEQ_CLS",
#     target_modules=["k_proj", "v_proj", "down_proj"],
#     feedforward_modules=["down_proj"]
# )



# config = PrefixTuningConfig(
#     task_type="SEQ_CLS",        # 序列分类任务
#     num_virtual_tokens=30,      # 前缀长度，可调
#     prefix_projection=True,     # 是否添加可训练投影层
#     inference_mode=False        # 是否是推理模式（训练时应为 False）
# )




# config = AdaLoraConfig(
#     task_type="SEQ_CLS",
#     r=8,                     # 初始秩
#     lora_alpha=16,
#     lora_dropout=0.1,
#     target_r=4,              # 最终目标秩
#     target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
#     beta1=0.85,
#     beta2=0.85,
#     tinit=200,               # 开始调整秩的迭代
#     tfinal=1000,             # 结束调整秩的迭代
#     deltaT=10,               # 秩更新间隔
#     total_step=2000,         # ✅ 新增参数！总训练步数（必须 > 0）
# )


from peft import PromptTuningConfig

config = PromptTuningConfig(
    task_type="SEQ_CLS",        # 你的任务类型
    num_virtual_tokens=40,      # prompt 长度，可根据经验调节 10~50
    token_dim=None,             # 默认使用模型 embedding 大小
    # encoder_hidden_size=None,   # 对某些模型可选
)



In [32]:
# len(tokenized_ds['train'])

In [34]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    # cache_dir=cache_dir,
    device_map="cuda:0",      # 强制 GPU
    # device_map="auto",
    # torch_dtype=torch.float16,       # 使用半精度
    # quantization_config=quantization_config,
    num_labels=labels.shape[1]
)



Loading checkpoint shards: 100%|██████████| 4/4 [00:04<00:00,  1.13s/it]
Some weights of LlamaForSequenceClassification were not initialized from the model checkpoint at /root/autodl-tmp/llm_models/LLM-Research/Meta-Llama-3.1-8B and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [35]:
model = prepare_model_for_kbit_training(model)
# model.gradient_checkpointing_disable() 
model = get_peft_model(model, config)
model.config.pad_token_id = tokenizer.pad_token_id


In [36]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir='multilabel_classification',
    learning_rate=1e-4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    num_train_epochs=40,
    eval_strategy="epoch",   
    save_strategy="epoch",   
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    # Prefix tuning does not work with gradient checkpointing
    # gradient_checkpointing=True,
    gradient_checkpointing=False,
    dataloader_pin_memory=False, 
    report_to="none",
)



In [37]:
tokenized_ds["test"]


Dataset({
    features: ['text', 'labels', 'input_ids', 'attention_mask'],
    num_rows: 371
})

In [38]:
print("🔍 检查数据集")
print("训练集样本数：", len(tokenized_ds["train"]))
print("验证集样本数：", len(tokenized_ds["val"]))

print("\n🔍 检查模型设备：", model.device)
print("LoRA 是否启用：", hasattr(model, "peft_config"))

print("\n🔍 检查模型参数")
print("trainable 参数数量：", sum(p.numel() for p in model.parameters() if p.requires_grad))
print("总参数数量：", sum(p.numel() for p in model.parameters()))

print("\n🔍 检查 GPU 可用性")
print("GPU 是否可用：", torch.cuda.is_available())
print("GPU 名称：", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "无")

# print("\n🔍 开始训练 ...")
# trainer.train()


🔍 检查数据集
训练集样本数： 3339
验证集样本数： 379

🔍 检查模型设备： cuda:0
LoRA 是否启用： True

🔍 检查模型参数
trainable 参数数量： 192512
总参数数量： 7505145856

🔍 检查 GPU 可用性
GPU 是否可用： True
GPU 名称： NVIDIA H800 PCIe


In [39]:
# print(tokenized_ds['train'][0]['labels'])
# print(len(tokenized_ds['train'][0]['labels']))
# # print(mlb.classes_)  # 如果用了 MultiLabelBinarizer
# # 检查原始数据集中标签的维度
# print("Train labels shape:", tokenized_ds["train"][0]["labels"], len(tokenized_ds["train"][0]["labels"]))
# print("Val labels shape:", tokenized_ds["val"][0]["labels"], len(tokenized_ds["val"][0]["labels"]))
# print("Test labels shape:", tokenized_ds["test"][0]["labels"], len(tokenized_ds["test"][0]["labels"]))


In [40]:
# from transformers import EarlyStoppingCallback

# trainer = CustomTrainer(
#     model=model,
#     args=training_args,
#     train_dataset=tokenized_ds['train'],
#     eval_dataset=tokenized_ds['test'],
#     processing_class=tokenizer,
#     data_collator=functools.partial(collate_fn, tokenizer=tokenizer),
#     compute_metrics=compute_metrics,
#     label_weights=torch.tensor(label_weights, device=model.device),
#     callbacks=[
#         EarlyStoppingCallback(
#             early_stopping_patience=5,      # 连续 3 次 eval 不提升就停
#             early_stopping_threshold=0.0    # 提升阈值
#         )
#     ]
# )


In [41]:
# train
trainer = CustomTrainer(
    model = model,
    args = training_args,
    train_dataset = tokenized_ds['train'],
    eval_dataset = tokenized_ds['test'],
    # tokenizer = tokenizer,
    processing_class=tokenizer,
    data_collator = functools.partial(collate_fn, tokenizer=tokenizer),
    compute_metrics = compute_metrics,
    label_weights = torch.tensor(label_weights, device=model.device),
)

trainer.train()

The model is already on multiple devices. Skipping the move to device specified in `args`.
LlamaForSequenceClassification will not detect padding tokens in `inputs_embeds`. Results may be unexpected if using padding tokens in conjunction with `inputs_embeds.`


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
trainer.evaluate(eval_dataset=tokenized_ds["test"])

In [ ]:
from sklearn.metrics import classification_report, accuracy_score, f1_score, precision_score, recall_score
import numpy as np
import torch

# === 预测测试集 ===
predictions = trainer.predict(tokenized_ds["val"])
preds = predictions.predictions
if isinstance(preds, torch.Tensor):
    preds = preds.cpu().numpy()

true_labels = predictions.label_ids

# === 将 logits 转为概率 (sigmoid)，再二值化 ===
pred_probs = 1 / (1 + np.exp(-preds))
pred_labels = (pred_probs > 0.5).astype(int)

# === 计算整体多标签指标 ===
micro_f1 = f1_score(true_labels, pred_labels, average='micro')
macro_f1 = f1_score(true_labels, pred_labels, average='macro')
weighted_f1 = f1_score(true_labels, pred_labels, average='weighted')
micro_precision = precision_score(true_labels, pred_labels, average='micro')
micro_recall = recall_score(true_labels, pred_labels, average='micro')

print("\n==== Overall Test Set Performance (Multilabel) ====")
print(f"Micro Precision: {micro_precision:.4f}")
print(f"Micro Recall:    {micro_recall:.4f}")
print(f"Micro F1:        {micro_f1:.4f}")
print(f"Macro F1:        {macro_f1:.4f}")
print(f"Weighted F1:     {weighted_f1:.4f}")

# === 输出每个标签的独立指标 ===
print("\n==== Per-label metrics ====")
num_labels = true_labels.shape[1]
for i in range(num_labels):
    label_name = label_names[i] if i < len(label_names) else f"Label_{i}"
    p = precision_score(true_labels[:, i], pred_labels[:, i], zero_division=0)
    r = recall_score(true_labels[:, i], pred_labels[:, i], zero_division=0)
    f1 = f1_score(true_labels[:, i], pred_labels[:, i], zero_division=0)
    acc = accuracy_score(true_labels[:, i], pred_labels[:, i])
    print(f"{label_name}: Precision={p:.4f}, Recall={r:.4f}, F1={f1:.4f}, Accuracy={acc:.4f}")



In [ ]:
# save model
# peft_model_id = 'multilabel_mistral'
# trainer.model.save_pretrained(peft_model_id)
# tokenizer.save_pretrained(peft_model_id)

In [ ]:
# plotting


In [ ]:
print(model_name,config)

In [ ]:
# from pathlib import Path
# from transformers import AutoTokenizer

# # 转换成 Path 对象，并把 \ 替换为 /
# peft_model_path = Path(r"D:\LoRA-MCD\ArmoRM_Llama3_8B").as_posix()

# tokenizer = AutoTokenizer.from_pretrained(
#     peft_model_path,
#     local_files_only=True
# )
